
# la Gerencia Comercial observa niveles continuos de cancelación de contratos al cierre de cada mes. No obstante, la toma de decisiones actual se realiza de manera reactiva, basada en impresiones generales o intuiciones del personal de servicio al cliente, sin un diagnóstico preciso respaldado por evidencia empírica sobre los patrones de uso, montos de facturación y servicios asociados a los usuarios que desertan.

### ¿como varia la tasa de abandono (churn rate) segun el tipo de contrato (contractype)y cual es la antiguedad promedio (tenure) critica con mayor numero de desercion?

# 1. Preparar el entorno e importar la clase de carga
El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar src/clases_po y se agrega src a sys.path. Así se puede importar HFDatasetLoader sin instalar el proyecto como paquete.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent
    
sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

/home/runner/work/PI3_Analitica_EA3/PI3_Analitica_EA3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Configurar el dataset y la salida
dataset_name identifica el repositorio en Hugging Face y split="train" indica la división que se descargará. El parámetro filename hace que el DataFrame también se guarde como CSV en src/sincronicas/datos/ cuando el notebook se ejecuta desde esa carpeta.

In [2]:
loader = HFDatasetLoader(
    dataset_name="Zainab-Jamil30/telecom-churn-dataset",
    split="train",
    filename="datos/telco_churn.csv",
)

# 3. Descargar el dataset y crear el DataFrame
El método load() descarga el dataset, lo convierte a pandas.DataFrame y lo asigna a la variable df. La expresión df.head() muestra las primeras filas para comprobar que la carga fue exitosa.

In [3]:
df = loader.load()
df.head(10)

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 10000/10000 [00:00<00:00, 448152.49 examples/s]

,CustomerID,Gender,Age,Tenure,ContractType,InternetService,MonthlyCharges,TotalCharges,PaymentMethod,TechSupport,StreamingTV,OnlineSecurity,PaperlessBilling,Churn
0,CUST00001,Male,68,16,Month-to-month,Fiber optic,63.46,1033.67,Electronic check,Yes,Yes,No,No,1
1,CUST00002,Female,57,51,Month-to-month,Fiber optic,79.29,4064.48,Mailed check,Yes,Yes,No,Yes,1
2,CUST00003,Male,24,11,Month-to-month,DSL,69.53,748.64,Bank transfer,Yes,No,No,Yes,1
3,CUST00004,Male,49,47,Month-to-month,Fiber optic,95.02,4527.19,Electronic check,Yes,No,No,Yes,1
4,CUST00005,Male,65,21,Month-to-month,Fiber optic,59.70,1272.96,Electronic check,No,Yes,Yes,Yes,1
5,CUST00006,Female,36,48,Month-to-month,DSL,72.66,3481.83,Electronic check,No,No,No,Yes,0
6,CUST00007,Male,23,51,Month-to-month,DSL,45.04,2344.69,Mailed check,Yes,No,No,No,0
7,CUST00008,Male,37,0,Month-to-month,Fiber optic,57.19,8.35,Mailed check,No,Yes,No,Yes,1
8,CUST00009,Male,44,65,Two year,DSL,52.57,3432.30,Electronic check,No,No,No,Yes,0
9,CUST00010,Female,70,63,Month-to-month,DSL,48.18,2951.96,Mailed check,No,No,Yes,Yes,0


# 4. Revisar la estructura del DataFrame
Esta celda realiza una revisión inicial: cuenta filas y columnas y muestra el tipo de cada columna junto con información de memoria y valores no nulos. Esta inspección ayuda a detectar problemas antes de analizar los datos.

In [4]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 10000
Columnas: 14
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerID        10000 non-null  object 
 1   Gender            10000 non-null  object 
 2   Age               10000 non-null  int64  
 3   Tenure            10000 non-null  int64  
 4   ContractType      10000 non-null  object 
 5   InternetService   10000 non-null  object 
 6   MonthlyCharges    10000 non-null  float64
 7   TotalCharges      10000 non-null  float64
 8   PaymentMethod     10000 non-null  object 
 9   TechSupport       10000 non-null  object 
 10  StreamingTV       10000 non-null  object 
 11  OnlineSecurity    10000 non-null  object 
 12  PaperlessBilling  10000 non-null  object 
 13  Churn             10000 non-null  int64  
dtypes: float64(2), int64(3), object(9)
memory usage: 1.1+ MB


se analiza valores maximos de cada campo y los tipos de internet instalados

In [5]:
contador = df.loc[df["Age"]].max()

print(contador)
print("="*50)
print(df["InternetService"].unique())

CustomerID             CUST00080
Gender                      Male
Age                           79
Tenure                        71
ContractType            Two year
InternetService               No
MonthlyCharges             95.61
TotalCharges             5644.38
PaymentMethod       Mailed check
TechSupport                  Yes
StreamingTV                  Yes
OnlineSecurity               Yes
PaperlessBilling             Yes
Churn                          1
dtype: object
['Fiber optic' 'DSL' 'No']


# Explicación de las Transformaciones Realizadas

Con el fin de facilitar el análisis exploratorio y la segmentación de clientes, se realizaron varias transformaciones sobre los datos originales. Estas transformaciones permiten generar nuevas variables que representan de una forma más clara las características de los usuarios y los servicios contratados.

## 1. Creación de la Categoría de Antigüedad

```python
df["CategoriaAntiguedad"] = pd.cut(...)
```

Esta transformación clasifica a los clientes según el tiempo que llevan en la compañía (`Tenure`).

Se definieron cuatro segmentos:

- Nuevo: entre 0 y 12 meses.
- Estable: entre 13 y 24 meses.
- Fiel: entre 25 y 48 meses.
- Muy Fiel: más de 48 meses.

El objetivo es facilitar el análisis de fidelización, ya que resulta más sencillo comparar grupos de clientes que interpretar directamente una cantidad de meses.

---

## 2. Creación del Grupo de Edad

```python
df["GrupoEdad"] = pd.cut(...)
```

La variable `Age` se transforma en categorías de edad.

Se definieron tres grupos:

- Joven.
- Adulto.
- Adulto Mayor.

Esta transformación permite analizar los comportamientos de consumo según grupos demográficos y no únicamente mediante edades individuales.

---

## 3. Conversión de Variables Sí/No a Valores Numéricos

```python
df[f"{col}_num"] = df[col].map({
    "Yes": 1,
    "No": 0
})
```

Las variables:

- TechSupport
- StreamingTV
- OnlineSecurity
- PaperlessBilling

contienen respuestas categóricas en formato texto.

Estas respuestas se transforman en:

- 1 = Sí
- 0 = No

El objetivo de esta transformación es facilitar cálculos posteriores, ya que muchos algoritmos de análisis y segmentación requieren variables numéricas.

---

## 4. Conversión del Servicio de Internet

```python
df["InternetService_num"]
```

La variable `InternetService` contiene diferentes tipos de conexión a internet.

Se realiza un agrupamiento donde:

- Fiber optic = 1
- DSL = 1
- No = 0

En este caso no interesa diferenciar el tipo de tecnología utilizada, sino identificar si el cliente cuenta o no con servicio de internet.

---

## 5. Creación de la Variable TotalServicios

```python
df["TotalServicios"]
```

Esta variable suma todos los servicios contratados por cada cliente.

Por ejemplo:

| Soporte | Streaming | Seguridad | Internet | Total |
|----------|----------|----------|----------|----------|
| 1 | 1 | 0 | 1 | 3 |
| 0 | 1 | 0 | 1 | 2 |

Su objetivo es medir el nivel de adopción de servicios por parte de cada cliente y facilitar la identificación de segmentos según la cantidad de productos contratados.

---

## 6. Creación de la Variable ServiciosContratados

```python
df["ServiciosContratados"]
```

Esta transformación consolida en una única columna los servicios activos de cada cliente.

Por ejemplo:

| Cliente | Resultado |
|-----------|-----------|
| Cliente 1 | Soporte Técnico, Internet |
| Cliente 2 | Streaming TV, Seguridad Online, Internet |
| Cliente 3 | Internet |

La finalidad de esta variable es hacer más comprensible la información para análisis descriptivos, reportes y visualizaciones, permitiendo identificar rápidamente qué servicios posee cada usuario.

---

# Conclusión

Las transformaciones realizadas permiten enriquecer el conjunto de datos mediante la creación de variables más interpretables y útiles para el análisis. Además, facilitan la aplicación de técnicas de segmentación y minería de datos al convertir información categórica en formatos adecuados para el procesamiento estadístico y los algoritmos de aprendizaje automático.

In [6]:
import pandas as pd

# esta transformacion genera un segmento según la fidelidad del usuario
df["CategoriaAntiguedad"] = pd.cut(
    df["Tenure"],
    bins=[0, 12, 24, 48, 100],
    labels=[
        "Nuevo",
        "Estable",
        "Fiel",
        "Muy Fiel"
    ],
    include_lowest=True
)
# se crea una columna con un segmento de edad
df["GrupoEdad"] = pd.cut(
    df["Age"],
    bins=[0, 30, 65, 80],
    labels=["Joven", "Adulto", "Adulto Mayor"]
)
# para las columnas que tienen un yes o no se reemplaza por valores de 0 y 1, para posteriores analisis de segmentos
columnas = [
    "TechSupport",
    "StreamingTV",
    "OnlineSecurity",
    "PaperlessBilling"
]

for col in columnas:

    df[f"{col}_num"] = df[col].map({
        "Yes": 1,
        "No": 0
    })


columnas1 = [
    "InternetService"
]
for col in columnas1:
    df[f"{col}_num"] = df[col].map({
        "Fiber optic": 1,
        'DSL': 1,
        "No": 0
    })
# se genera una nueva columna que suma el total de servicios y tener el valor para crear segmentos
df["TotalServicios"] = (
    df["TechSupport_num"] +
    df["StreamingTV_num"] +
    df["OnlineSecurity_num"] +
    df["InternetService_num"]
)
# se genera una nueva columna que consolida los servicios contratados
servicios = {
    "TechSupport_num": "Soporte Técnico",
    "StreamingTV_num": "Streaming TV",
    "OnlineSecurity_num": "Seguridad Online",
    "InternetService_num": "Internet"
}

df["ServiciosContratados"] = (
    df[list(servicios.keys())]
    .apply(
        lambda fila: ", ".join(
            servicios[col]
            for col in fila.index
            if fila[col] == 1
        ),
        axis=1
    )
)

df.info()
df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   CustomerID            10000 non-null  object  
 1   Gender                10000 non-null  object  
 2   Age                   10000 non-null  int64   
 3   Tenure                10000 non-null  int64   
 4   ContractType          10000 non-null  object  
 5   InternetService       10000 non-null  object  
 6   MonthlyCharges        10000 non-null  float64 
 7   TotalCharges          10000 non-null  float64 
 8   PaymentMethod         10000 non-null  object  
 9   TechSupport           10000 non-null  object  
 10  StreamingTV           10000 non-null  object  
 11  OnlineSecurity        10000 non-null  object  
 12  PaperlessBilling      10000 non-null  object  
 13  Churn                 10000 non-null  int64   
 14  CategoriaAntiguedad   10000 non-null  category
 15  Gru

,CustomerID,Gender,Age,Tenure,ContractType,InternetService,MonthlyCharges,TotalCharges,PaymentMethod,TechSupport,...,Churn,CategoriaAntiguedad,GrupoEdad,TechSupport_num,StreamingTV_num,OnlineSecurity_num,PaperlessBilling_num,InternetService_num,TotalServicios,ServiciosContratados
0,CUST00001,Male,68,16,Month-to-month,Fiber optic,63.46,1033.67,Electronic check,Yes,...,1,Estable,Adulto Mayor,1,1,0,0,1,3,"Soporte Técnico, Streaming TV, Internet"
1,CUST00002,Female,57,51,Month-to-month,Fiber optic,79.29,4064.48,Mailed check,Yes,...,1,Muy Fiel,Adulto,1,1,0,1,1,3,"Soporte Técnico, Streaming TV, Internet"
2,CUST00003,Male,24,11,Month-to-month,DSL,69.53,748.64,Bank transfer,Yes,...,1,Nuevo,Joven,1,0,0,1,1,2,"Soporte Técnico, Internet"
3,CUST00004,Male,49,47,Month-to-month,Fiber optic,95.02,4527.19,Electronic check,Yes,...,1,Fiel,Adulto,1,0,0,1,1,2,"Soporte Técnico, Internet"
4,CUST00005,Male,65,21,Month-to-month,Fiber optic,59.70,1272.96,Electronic check,No,...,1,Estable,Adulto,0,1,1,1,1,3,"Streaming TV, Seguridad Online, Internet"
5,CUST00006,Female,36,48,Month-to-month,DSL,72.66,3481.83,Electronic check,No,...,0,Fiel,Adulto,0,0,0,1,1,1,Internet
6,CUST00007,Male,23,51,Month-to-month,DSL,45.04,2344.69,Mailed check,Yes,...,0,Muy Fiel,Joven,1,0,0,0,1,2,"Soporte Técnico, Internet"
7,CUST00008,Male,37,0,Month-to-month,Fiber optic,57.19,8.35,Mailed check,No,...,1,Nuevo,Adulto,0,1,0,1,1,2,"Streaming TV, Internet"
8,CUST00009,Male,44,65,Two year,DSL,52.57,3432.30,Electronic check,No,...,0,Muy Fiel,Adulto,0,0,0,1,1,1,Internet
9,CUST00010,Female,70,63,Month-to-month,DSL,48.18,2951.96,Mailed check,No,...,0,Muy Fiel,Adulto Mayor,0,0,1,1,1,2,"Seguridad Online, Internet"


# 6. analisis pregunta de negocio :

## ¿como varia la tasa de abandono (churn rate) segun el tipo de contrato (contractype)y cual es la antiguedad promedio (tenure) critica con mayor numero de desercion?

In [7]:
(
    df[df["Churn"] == 1]
    ["ContractType"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

ContractType
Month-to-month    85.53
One year           9.08
Two year           5.39
Name: proportion, dtype: float64

In [8]:
resultado = (
    df[df["Churn"] == 1]
    .groupby("ContractType")
    .agg(
        Cantidad_Clientes=("Churn", "count"),
        Tenure_Promedio=("Tenure", "mean"),
        Tenure_Minimo=("Tenure", "min"),
        Tenure_Maximo=("Tenure", "max")
    )
)

resultado["Porcentaje"] = (
    resultado["Cantidad_Clientes"]
    / resultado["Cantidad_Clientes"].sum()
    * 100
).round(2)

resultado = resultado[
    [
        "Cantidad_Clientes",
        "Porcentaje",
        "Tenure_Promedio",
        "Tenure_Minimo",
        "Tenure_Maximo"
    ]
].sort_values(
    "Porcentaje",
    ascending=False
)

display(resultado.round(2))

,Cantidad_Clientes,Porcentaje,Tenure_Promedio,Tenure_Minimo,Tenure_Maximo
ContractType,,,,,
Month-to-month,3900,85.53,32.93,0,71
One year,414,9.08,27.39,0,71
Two year,246,5.39,27.16,0,71


### Interpretación de Resultados

La mayoría de los clientes que abandonaron el servicio tenían contratos de tipo **Month-to-month** (mensual). Este resultado indica que los usuarios con contratos de corta duración presentan una mayor probabilidad de cancelar el servicio en comparación con aquellos que poseen contratos de uno o dos años.

Adicionalmente, se observó que los clientes con contratos mensuales presentan una antigüedad promedio (**Tenure**) menor que la de los clientes con contratos de mayor duración. Esto sugiere que los usuarios más recientes o con menor compromiso contractual son más propensos a abandonar la compañía.

Por el contrario, los clientes con contratos de uno o dos años muestran una permanencia promedio más alta y una menor participación dentro del grupo de usuarios que cancelaron el servicio.

Estos hallazgos permiten inferir que la duración del contrato y la antigüedad del cliente tienen una relación importante con el fenómeno de abandono (**Churn**), evidenciando que los clientes con menor nivel de compromiso contractual representan un segmento con mayor riesgo de cancelación.


7. Importar la clase de reportes
Reportes recibe un DataFrame en sus métodos y permite generar un resumen textual, imprimirlo en la consola o guardarlo en un archivo .txt.

In [9]:
from clases_po import Reportes

8. Generar y visualizar el reporte como texto
generar_reporte(df) devuelve un str con el número de filas, columnas, duplicados, tipos de datos, valores nulos y estadísticas numéricas. En esta celda el texto se guarda temporalmente en texto y se imprime.

In [10]:
reportes = Reportes()
texto = reportes.generar_reporte(df)

print(texto)

REPORTE DEL DATAFRAME
Filas: 10000
Columnas: 23
Duplicados: 0

Columnas y tipos:
- CustomerID: object
- Gender: object
- Age: int64
- Tenure: int64
- ContractType: object
- InternetService: object
- MonthlyCharges: float64
- TotalCharges: float64
- PaymentMethod: object
- TechSupport: object
- StreamingTV: object
- OnlineSecurity: object
- PaperlessBilling: object
- Churn: int64
- CategoriaAntiguedad: category
- GrupoEdad: category
- TechSupport_num: int64
- StreamingTV_num: int64
- OnlineSecurity_num: int64
- PaperlessBilling_num: int64
- InternetService_num: int64
- TotalServicios: int64
- ServiciosContratados: object

Valores nulos por columna:
- CustomerID: 0
- Gender: 0
- Age: 0
- Tenure: 0
- ContractType: 0
- InternetService: 0
- MonthlyCharges: 0
- TotalCharges: 0
- PaymentMethod: 0
- TechSupport: 0
- StreamingTV: 0
- OnlineSecurity: 0
- PaperlessBilling: 0
- Churn: 0
- CategoriaAntiguedad: 0
- GrupoEdad: 0
- TechSupport_num: 0
- StreamingTV_num: 0
- OnlineSecurity_num: 0
- Pape

In [11]:
descriptivo = reportes.generar_reporte_descriptivo(df)
print(descriptivo)

REPORTE DESCRIPTIVO DEL DATAFRAME

1. DIMENSIONES DEL DATASET
- Número de filas: 10000
- Número de columnas: 23

2. MUESTRA DE REGISTROS (primeras 5 filas)
  CustomerID  Gender  Age  Tenure    ContractType InternetService  MonthlyCharges  TotalCharges     PaymentMethod TechSupport StreamingTV OnlineSecurity PaperlessBilling  Churn CategoriaAntiguedad     GrupoEdad  TechSupport_num  StreamingTV_num  OnlineSecurity_num  PaperlessBilling_num  InternetService_num  TotalServicios                      ServiciosContratados
0  CUST00001    Male   68      16  Month-to-month     Fiber optic           63.46       1033.67  Electronic check         Yes         Yes             No               No      1             Estable  Adulto Mayor                1                1                   0                     0                    1               3   Soporte Técnico, Streaming TV, Internet
1  CUST00002  Female   57      51  Month-to-month     Fiber optic           79.29       4064.48      Mailed chec

9. Imprimir el reporte directamente
imprimir_reporte(df) ejecuta la generación y muestra el resultado directamente. Es útil para consultar rápidamente el resumen durante una ejecución interactiva.

In [12]:
reportes.imprimir_reporte(df)



REPORTE DEL DATAFRAME
Filas: 10000
Columnas: 23
Duplicados: 0

Columnas y tipos:
- CustomerID: object
- Gender: object
- Age: int64
- Tenure: int64
- ContractType: object
- InternetService: object
- MonthlyCharges: float64
- TotalCharges: float64
- PaymentMethod: object
- TechSupport: object
- StreamingTV: object
- OnlineSecurity: object
- PaperlessBilling: object
- Churn: int64
- CategoriaAntiguedad: category
- GrupoEdad: category
- TechSupport_num: int64
- StreamingTV_num: int64
- OnlineSecurity_num: int64
- PaperlessBilling_num: int64
- InternetService_num: int64
- TotalServicios: int64
- ServiciosContratados: object

Valores nulos por columna:
- CustomerID: 0
- Gender: 0
- Age: 0
- Tenure: 0
- ContractType: 0
- InternetService: 0
- MonthlyCharges: 0
- TotalCharges: 0
- PaymentMethod: 0
- TechSupport: 0
- StreamingTV: 0
- OnlineSecurity: 0
- PaperlessBilling: 0
- Churn: 0
- CategoriaAntiguedad: 0
- GrupoEdad: 0
- TechSupport_num: 0
- StreamingTV_num: 0
- OnlineSecurity_num: 0
- Pape

In [13]:
reportes.imprimir_reporte(df)

REPORTE DEL DATAFRAME
Filas: 10000
Columnas: 23
Duplicados: 0

Columnas y tipos:
- CustomerID: object
- Gender: object
- Age: int64
- Tenure: int64
- ContractType: object
- InternetService: object
- MonthlyCharges: float64
- TotalCharges: float64
- PaymentMethod: object
- TechSupport: object
- StreamingTV: object
- OnlineSecurity: object
- PaperlessBilling: object
- Churn: int64
- CategoriaAntiguedad: category
- GrupoEdad: category
- TechSupport_num: int64
- StreamingTV_num: int64
- OnlineSecurity_num: int64
- PaperlessBilling_num: int64
- InternetService_num: int64
- TotalServicios: int64
- ServiciosContratados: object

Valores nulos por columna:
- CustomerID: 0
- Gender: 0
- Age: 0
- Tenure: 0
- ContractType: 0
- InternetService: 0
- MonthlyCharges: 0
- TotalCharges: 0
- PaymentMethod: 0
- TechSupport: 0
- StreamingTV: 0
- OnlineSecurity: 0
- PaperlessBilling: 0
- Churn: 0
- CategoriaAntiguedad: 0
- GrupoEdad: 0
- TechSupport_num: 0
- StreamingTV_num: 0
- OnlineSecurity_num: 0
- Pape

10. Guardar el reporte en un archivo de texto
guardar_reporte(df, ruta) crea los directorios necesarios y escribe el reporte en UTF-8. El archivo quedará en src/sincronicas/reportes/telco_churn.txt cuando se ejecute desde src/sincronicas.

In [14]:
ruta = reportes.guardar_reporte(
    df,
    "reportes/telco_churn.txt",
)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/telco_churn.txt


11. se guarda el reporte descriptivo a nivel de los demas reportes

In [15]:
ruta = Path("reportes/Reporte_descriptivo_telco_churn.txt")
reportes.guardar_reporte_descriptivo(df, ruta)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/Reporte_descriptivo_telco_churn.txt
